# CIFAR-100 random subset 4 of 5

This notebook trains a fresh ten-class ResNet-18 for draw 4 (seed 3). All five draws and their class mappings are written to the experiment plan before training. The 25-per-class calibration track is nested in the 100-per-class track. Both are disjoint from training, and the policies freeze before the official test data is loaded for scoring.

Install `requirements-training.txt` in the kernel environment. Set `SDM_CIFAR100_DATA` to the directory containing the prepared `cifar-100-python` folder, or explicitly set the download flag below. Smoke is the default. Set `SDM_SUBSET_BENCHMARK=1` before starting Jupyter for the fixed 40-epoch protocol. Use a fresh `SDM_SUBSET_OUTPUT` for a repeated completed draw. Smoke and benchmark evidence are stored in separate subdirectories.

In [ ]:
import os
from pathlib import Path
import sys

REPO = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'experiments' / 'cifar100_subsets.py').is_file())
sys.path.insert(0, str(REPO / 'experiments'))
from cifar100_subsets import OUTPUT

SMOKE = os.environ.get('SDM_SUBSET_BENCHMARK') != '1'
OUTPUT_ROOT = Path(os.environ.get('SDM_SUBSET_OUTPUT', OUTPUT)).expanduser()

from cifar100_subsets import run_draw
import pandas as pd

DRAW = 4
DATA_DIR = Path(os.environ.get('SDM_CIFAR100_DATA', REPO / 'data' / 'cifar100')).expanduser()
DOWNLOAD = False
print(f"Draw {DRAW} | {'smoke' if SMOKE else 'benchmark'} | {OUTPUT_ROOT}")


## Compare frozen policies

Raw and temperature softmax use the largest whole-score-group admission rule that reaches 95% empirical calibration accuracy with at least 100 admissions. SDM uses the identity-adaptor high-reliability region at alpha 0.95. These policies can admit different images. Zero admissions leave precision undefined. Metrics exclude test images whose pixels exactly match training or calibration images within this draw. The export retains their predictions and flags.

In [ ]:
rows = run_draw(DRAW, data_dir=DATA_DIR, output=OUTPUT_ROOT, smoke=SMOKE, download=DOWNLOAD)
pd.DataFrame(rows)[['track', 'method', 'official_examples', 'excluded_seen_pixels', 'examples', 'admitted', 'correct_admitted', 'coverage', 'accepted_accuracy']]

Use [the aggregate notebook](cifar100_random_subset_summary.ipynb) to compare all five draws. Results apply to the sampled classes only. Prior inspection of the CIFAR-100 test set and class overlap across draws limit confirmatory interpretation. Smoke outputs are pipeline checks.